# Загрузка, анализ, подготовка данных

## Задача

После разбора домашнего задания выяснелось, что данные были зашумлены. Вот некоторые из признаков проблемных переводов:
1) Длинна перевода больше длинны оргинала более чем в $n$ раз (и наоборот) Где $n$ определяется эмперически для каждого датасета;
2) В паре примера одно из полей (`src` и/или `dst`) пустое;
3) В переводе и оригинале используются арабские цифры, но они различаются;
4) в паре используются скобки и кавычки, они различаются.

## Подготовка среды

### Импорт библиотек

In [212]:
import random
import string
import re

import pandas as pd
import numpy as np
import plotly.express as px

from datasets import load_dataset, Features, Value, concatenate_datasets, Dataset, DatasetDict

### Полезные функции

In [213]:
# Функция для подсчета уникальных символов
def count_unique_characters_in_split(dataset, split_name, feature_name):
    unique_characters = set()
    for example in dataset[split_name][feature_name]:
        if example is not None:
            unique_characters.update(example)
    num_unique_characters = len(unique_characters)

    print(f"Number of unique characters in {split_name} split: {num_unique_characters}")
    print(f"Unique characters in {split_name} split: {unique_characters}", end="\n\n")

    return num_unique_characters, unique_characters

# Функция для подсчета отношения длин src и dst и рисования гистограммы
def plot_length_difference_distribution(df, split_name='train', bins=100):
    df['length_difference'] = df['src'].apply(len) / df['dst'].apply(len)
    fig = px.histogram(df, x='length_difference', nbins=bins, title=f'Распределение разницы в длине между src и dst для {split_name}')
    
    tick_step = 0.25
    x_range = [0, 3.5]
    tickvals = np.arange(x_range[0], x_range[1] + tick_step, tick_step)
    
    fig.update_xaxes(title='Отношение длин', range=x_range, tickvals=tickvals)

    fig.show()

## Загрузка данных

Загрузим данные и посмотрим на случайно выбранный примеры из каждой части датасета:

In [214]:
data_files = {
    "train": "raw_train.jsonl",
    "validation": "raw_val.jsonl",
    "test": "raw_test_no_reference.jsonl"
}

features = Features({
    'src': Value('string'),
    'dst': Value('string'),
})

dataset = load_dataset("json", data_files=data_files, features=features)
dataset

DatasetDict({
    train: Dataset({
        features: ['src', 'dst'],
        num_rows: 300000
    })
    validation: Dataset({
        features: ['src', 'dst'],
        num_rows: 500
    })
    test: Dataset({
        features: ['src', 'dst'],
        num_rows: 1000
    })
})

In [215]:
example_index = random.randint(0, len(dataset['validation']) - 1)

print(f"Train example: {dataset['train'][example_index]}, src lengh {len(dataset['train'][example_index]['src'])}, dst lengh {len(dataset['train'][example_index]['dst'])}")
print(f"Validation example: {dataset['validation'][example_index]}, src lengh {len(dataset['validation'][example_index]['src'])}, dst lengh {len(dataset['validation'][example_index]['dst'])}")
print(f"Test example: {dataset['test'][example_index]}, src lengh {len(dataset['test'][example_index]['src'])}")

Train example: {'src': '◢◠▻◠ ◧ ◉◪▦◪▦◗■ ▫◠◎◠◎ ◎▪?', 'dst': 'You shut your whore mouth, okay?'}, src lengh 24, dst lengh 32
Validation example: {'src': "◆◭▦▴▽◀◠▫▪ ◀◇▱◕▴▱◪◓◗▦◈▴■ ◇▢▴▱▱◗▨▱◪ ○◓◗▢◧▦◠'▦▪▦ ◀◭◳◭▨ ◀◣▱◭◎▩▦◈◪■ 2 ◫▱◠ 4 ◫▦◉ ◀▩▽▩▨▱▩◐◭▦◪ ◚◠◓◠◀◗▱▴▼▴▨ ▨◠◈◠◓ ▽◠◐◬◒ ◀◪▨▱◪▦◫◳◂◓▵", 'dst': 'In some areas of the southwestern United States, mainly in most of Arizona, up to 5-10 cm of precipitation is expected.'}, src lengh 122, dst lengh 119
Test example: {'src': '▭◠◓◗◞■ ○◞◞◂▼◫◠▫◪◈ □◓◪◞◞\'▴ ◚◪◓◈◫◐◗ ◓◇▻◂◓▫◠▸◈◠ ◂ ◠▦▱◠◓▪ ◒◇▽▱◪ ◠▨▫◠◓◈◬: "◢▴▦◈◗◎◗▢◗ ▨◨◓▫◠◓◎◠▨ ◫◉◗▦ ▢◠◎◠▦▪◎▪▢ ◳◂▨▫◨▵', 'dst': None}, src lengh 111


Судя по выводу, данные загрузились без проблем с кодировкой.

## Анализ датасета

### Анализ символов алфавита

Изучим алфавит данного языка, посмотрим на уникальные символы и их количество для признака `src`:

In [216]:
feature = 'src'

split_name = 'train'
train_num_unique_chars_src, train_unique_chars_src = count_unique_characters_in_split(dataset, split_name, feature)


split_name = 'validation'
validation_num_unique_chars_src, validation_unique_chars_src = count_unique_characters_in_split(dataset, 'validation', feature)


split_name = 'test'
test_num_unique_chars_src, test_unique_chars_src = count_unique_characters_in_split(dataset, 'test', feature)

Number of unique characters in train split: 176
Unique characters in train split: {'◐', '◤', '▪', '\u202d', '▯', 'ţ', '◖', '¡', '◌', 'þ', '´', 'á', '◦', '◓', '~', '▴', '◉', '#', '◱', 'ª', 'ß', '▬', '§', 'ă', 'Î', '◭', '*', '`', '!', '2', '•', '▸', '[', '¿', '◬', '◁', "'", '◲', '+', 'â', '▹', '£', '─', '◢', 'Ä', '▲', '◩', '▵', '3', '=', 'Þ', 'ú', ' ', '—', '◀', ')', '●', 'ν', '◛', '○', 'ó', 'ô', '▫', 'û', '◂', ';', '‚', 'é', 'ä', '\x99', '▷', '◯', '7', 'đ', '6', 'ð', 'å', '"', 'í', '◈', '▼', 'Â', '▶', '◑', '♫', '◥', 'ë', '¶', '◝', 'Ý', '◒', 'ι', '_', '▿', '◗', '▨', '¤', '▻', '◎', '►', 'î', ']', 'ý', '\u200b', '▱', '△', '{', '–', '@', 'ø', '$', '▾', 'ï', '%', '▭', '▦', 'è', '◄', '4', '\\', '◙', '◚', '◊', '0', '▮', '▽', '▥', '◪', '◨', '◅', '■', '◰', '^', '◟', '™', '(', '°', '◳', 'É', 'ο', '▰', '◇', '}', 'º', '♪', '◜', '◘', '◣', '◞', '1', '◕', 'ñ', '◫', '▩', '◆', '?', '◠', '◍', '9', '▣', '◮', '▢', '8', '◔', '◃', ':', '\x9d', '\x9e', '/', '\xa0', '□', '▧', '◡', '◧', '▤', '5'}

Number of uni

Тренировочная часть по количеству символов резко отличается от валидационной и тестовой. Вероятно она сильно зашумлена лишними символами. Необходима чистка данных. Сделаем то же для признака `dst`:

In [217]:
feature = 'dst'

split_name = 'train'
train_num_unique_chars_dst, train_unique_chars_dst = count_unique_characters_in_split(dataset, split_name, feature)


split_name = 'validation'
validation_num_unique_chars_dst, validation_unique_chars_dst = count_unique_characters_in_split(dataset, 'validation', feature)

Number of unique characters in train split: 198
Unique characters in train split: {'ò', 'B', 'l', 'h', '©', 'Æ', 'ì', '\u202d', 'þ', '¡', 'b', '´', 'á', '~', 'у', 'ª', '#', 'H', 'Ã', '±', 'œ', 't', 'Μ', 'À', '§', 'ă', 'Ü', '*', '`', '!', '2', '¯', 'V', '′', 'W', '[', '¿', '鈥', '\xad', 'с', 'w', "'", 'ư', 'Ο', '+', 'â', 'y', 'S', '£', '─', '\x83', 'I', 'Ä', ',', 'ç', '³', '3', '˜', '=', 'ú', ' ', 'Y', 'Ι', '\x8d', '—', '¢', 'İ', 'p', ')', '●', 'ν', 'ί', 'u', 's', '¬', '│', 'L', 'ü', 'g', 'ó', 'ô', 'ê', 'J', '.', ';', '\x97', 'D', '‚', 'ş', 'f', 'P', 'é', 'o', 'ä', 'O', '\x99', '\x90', 'Ν', 'Η', 'υ', 'Ö', '7', 'q', '6', 'í', '"', 'å', 'ð', 'Ż', 'C', 'Ë', 'Â', 'v', '♫', 'ë', '€', '¶', '½', 'z', 'Ý', '_', '¤', 'Q', 'Β', ']', 'ý', 'a', 'F', '{', '–', 'ρ', '@', '$', '-', 'ï', '%', 'Τ', 'à', 'R', 'A', 'æ', 'T', 'ﬁ', 'è', 'Ð', '4', '\\', 'N', 'ﬂ', '‒', '檛', 'ē', '0', 'r', '^', '″', '™', 'k', 'i', 'ã', '(', '°', 'É', 'ο', '\x92', 'ı', 'd', 'U', 'n', '}', 'ć', '♪', 'X', 'ö', '1', 'c', 'G', 'õ', 

В этом признаке разница еще больше. Что подтверждает необходимость чистки данных.

### Признак `src`

Будем считать что тестовая и валидационная части датасета чистые, так как в них меньше символов. Найдём мусорные символы в признаке `src`: 
1) Обьединим используемые в признаке `src` множества символов тестовой и валидационной частей датасета. Добавим в это множество все арабские цифры. Далее будем называть это множество эталонным.
2) Найдём разность между эталонным и тренировочным множеством. Так мы получим символы которые необходимо удалить из тренировочной части датасета.

In [219]:
arabic_digits = {'0', '1', '2', '3', '4', '5', '6', '7', '8', '9'}
etalon_unique_chars_src = test_unique_chars_src | validation_unique_chars_src | arabic_digits

chars_to_remove_src = train_unique_chars_src - etalon_unique_chars_src

print(f"Characters to remove from train split: {chars_to_remove_src}", end="\n\n")

Characters to remove from train split: {'Â', 'Þ', 'ú', '♫', 'ë', '¶', '—', '^', '\u202d', 'ţ', 'þ', '¡', '™', '´', '●', 'Ý', 'ν', 'ι', '_', '~', 'ª', '°', '¤', 'É', 'ο', 'î', 'ß', '§', 'ă', 'ó', 'ô', 'Î', 'ý', '\u200b', 'û', '}', 'º', '*', '`', '{', '♪', '–', '•', '‚', '¿', '$', 'ñ', 'ï', 'ä', 'ø', '\x99', '\x9d', '\x9e', 'è', '£', '─', '\xa0', 'đ', '\\', 'Ä', 'ð', 'å', '='}



### Признак `dst`

Будем считать что валидационная часть датасета чистая. Найдём символы которые есть в тестовой части, но нет тренировочной: 

In [220]:
chars_to_remove_dst = train_unique_chars_dst - validation_unique_chars_dst

print(f"Characters to remove from train split: {chars_to_remove_dst}", end="\n\n")

Characters to remove from train split: {'ò', '©', 'Æ', 'ì', '\u202d', 'þ', '¡', '´', 'á', '~', 'у', 'ª', '#', 'Ã', '±', 'œ', 'Μ', 'À', '§', 'ă', 'Ü', '*', '`', '¯', '′', '[', '¿', '鈥', '\xad', 'с', 'ư', 'Ο', 'â', '─', '\x83', 'Ä', 'ç', '³', '˜', '=', 'ú', '¢', 'Ι', '\x8d', 'İ', '●', 'ν', 'ί', '¬', '│', 'ó', 'ô', 'ê', '\x97', '‚', 'ş', 'ä', '\x99', '\x90', 'Ν', 'Η', 'υ', 'Ö', 'ð', 'í', 'å', 'Ż', 'Ë', 'Â', '♫', 'ë', '€', '¶', '½', 'Ý', '¤', 'Β', ']', 'ý', '{', 'ρ', 'ï', 'Τ', 'à', 'æ', 'ﬁ', 'è', 'Ð', 'ﬂ', '‒', '檛', 'ē', '^', '″', '™', 'ã', '°', 'É', 'ο', '\x92', 'ı', '}', 'ć', '♪', 'ö', 'õ', 'ñ', 'ø', '\x9d', '\xa0', 'š', 'Á'}



В множестве символов встречаются символы арихметических операций (`+`,`-`,`*`,`/`,`=`), посмотрим на примеры которые их содержат:

In [221]:
# df = pd.DataFrame(dataset['train'])

# # pattern = re.compile(r'[+\-*/=]')
# pattern = re.compile(r'[&]')
# filtered_df = df[df['src'].apply(lambda x: bool(pattern.search(x)))]
# filtered_df

df = pd.DataFrame(dataset['validation'])

# pattern = re.compile(r'[+\-*/=]')
pattern = re.compile(r'[\[]')
filtered_df = df[df['src'].apply(lambda x: bool(pattern.search(x)))]
filtered_df

,src,dst
180,◟◪▦◗ 17▵ ◀◣▱◕◪◈▴▨◫ ◳◠◓◬◒ ◠▦◠ ▻◠◓▫◫ ◍◫▦◠▦◞◎◠▦▪ ...,The race in the new 17th district has sparked ...
190,◄▴▼▱◗◞ ▩▽◪▱▴◓◫▦◫▦ ▾▦◚◠▦▱◠◓▪▽▱◠ ◫▱◕◫▱◫ ◧▱◠◓◠▨ ◢...,"As for the name of the Assembly's members, the..."


Судя по всему стоит оставить только символ `-`, так как он означает не только минус или отрицание, но и прямую речь. Для остальных символов очень мало примеров. Оставим так же символ `/`.

## Очистка данных

Объединим два множества символов которые необходимо удалить. Еще раз проверим, чтобы в нём не оказалось букв, цифр и символов `-` и `/`:

In [ ]:
chars_to_remove = (chars_to_remove_dst | chars_to_remove_src) - set(string.ascii_letters + string.digits)

# Удаление элементов из множества
chars_to_remove.difference_update({'#', '&', '$', '[', ']', '½', 'á', 'â', '“', '”', '\\', '–', '—',})

print(len(chars_to_remove), chars_to_remove)

Очистим тренировочную часть датасета от лишних символов:

In [ ]:
def clear_example(example):
    for char in chars_to_remove:
        for feature in example:
            example[feature] = example[feature].replace(char, '') 

    return example

dataset['train'] = dataset['train'].map(clear_example, num_proc=10)

Проверим сколько теперь символов в частях датасета:

In [ ]:
feature = 'src'

split_name = 'train'
train_num_unique_chars_src, train_unique_chars_src = count_unique_characters_in_split(dataset, split_name, feature)

split_name = 'validation'
validation_num_unique_chars_src, validation_unique_chars_src = count_unique_characters_in_split(dataset, 'validation', feature)


split_name = 'test'
test_num_unique_chars_src, test_unique_chars_src = count_unique_characters_in_split(dataset, 'test', feature)

Теперь тренировочный датасете отличается от тестового и валидационного по признаку `src` на следующие символы:

In [ ]:
validation_unique_chars_src - train_unique_chars_src 

In [ ]:
test_unique_chars_src - train_unique_chars_src

Посмотрим на признак `dst`:

In [ ]:
feature = 'dst'

split_name = 'train'
train_num_unique_chars_src, train_unique_chars_src = count_unique_characters_in_split(dataset, split_name, feature)

split_name = 'validation'
validation_num_unique_chars_src, validation_unique_chars_src = count_unique_characters_in_split(dataset, 'validation', feature)

Разница в используемых символах:

In [ ]:
validation_unique_chars_src - train_unique_chars_src

## Поиск некачественных примеров данных 

### Примеры с большой разницей в длинне

Преобразуем все части датасета в Dataframe, для более удобной работы c графиками:

In [ ]:
df_train = pd.DataFrame(dataset['train'])
df_validation = pd.DataFrame(dataset['validation'])
df_train.head()

Посмотрим, во сколько раз отличаются по длинне признаки примеров `src` и `dst` в тренировочной и валидационной частях датасета:

In [ ]:
plot_length_difference_distribution(df_train, 'train', bins=300)

In [ ]:
plot_length_difference_distribution(df_validation, 'validation', bins=300)

Хорошо видно, что большинство примеров валидации укладывается в диапазон от 0.5 до 1.75 включительно. Что говорит о том, что наш язык символьный и в тренировочной части датасета ~ 20k примеров можно выкинуть. Удалим все примеры соотношение длин признаков которых менее 0.5 и более 1.75:

In [ ]:
df_train = df_train[(df_train['length_difference'] >= 0.5) & (df_train['length_difference'] <= 1.75)]
df_train = df_train_filtered.drop(columns=['length_difference', '__index_level_0__'], errors='ignore')

df_train

### Примеры в которых различается порядок и количество цифр

Проверим наличие примеров в которых различается количество и порядок цифр:

In [ ]:
def extract_digits(text):
    return ''.join(re.findall(r'\d', text))

df_train['src_digits'] = df_train['src'].apply(extract_digits)
df_train['dst_digits'] = df_train['dst'].apply(extract_digits)

digit_mismatch_df = df_train[df_train['src_digits'] != df_train['dst_digits']]

digit_mismatch_df[['src', 'dst', 'src_digits', 'dst_digits']]

Удалим найденые битые примеры:

In [ ]:
# Remove examples with digit mismatches
df_train_filtered = df_train[df_train['src_digits'] == df_train['dst_digits']]

# Drop the auxiliary columns used for filtering
df_train_filtered = df_train_filtered.drop(columns=['length_difference', 'src_digits', 'dst_digits'], errors='ignore')

df_train = df_train_filtered
df_train

### Проблемы с прямой речью

Посмотрим на примеры которые начинаются с символа `►` в `src` и не начинаются с символа `-` или `—` в `dst`:

In [ ]:
filtered_examples = df_train[(df_train['src'].str.startswith('►')) & (~df_train['dst'].str.startswith(('-', '—')))]
filtered_examples

Это примеры с поврежденной прямой речью, удалим их:

In [ ]:
df_train = df_train[~((df_train['src'].str.startswith('►')) & (~df_train['dst'].str.startswith(('-', '—'))))]
df_train

Посмотрим на обратную ситуацию, когда в `dst` есть символы `-` или `—`, но в `src` нет символа `►` в начале строки:  

In [ ]:
filtered_examples_reverse = df_train[(df_train['dst'].str.startswith(('-', '—'))) & (~df_train['src'].str.startswith('►'))]
filtered_examples_reverse

Судя по всему это тоже битая прямая речь, удалим ее:

In [ ]:
df_train = df_train[~((df_train['dst'].str.startswith(('-', '—'))) & (~df_train['src'].str.startswith('►')))]
df_train

### Примеры в которых различаются символы пунктуации

Проверим наличие примеров в которых в признаках различаются символы пунктуации:

In [ ]:
# # Функция для подсчета различий в пунктуации
# def count_punctuation_differences(row):
#     src_punctuation = row['src_punctuation']
#     dst_punctuation = row['dst_punctuation']
#     differences = sum(1 for a, b in zip(src_punctuation, dst_punctuation) if a != b)
#     differences += abs(len(src_punctuation) - len(dst_punctuation))
#     return differences

# # Применение функции к DataFrame и фильтрация строк с более чем тремя различиями в пунктуации
# punctuation_mismatch_df['punctuation_differences'] = punctuation_mismatch_df.apply(count_punctuation_differences, axis=1)
# punctuation_mismatch_df_filtered = punctuation_mismatch_df[punctuation_mismatch_df['punctuation_differences'] > 3]

# # Вывод всех примеров, в которых различается больше трёх символов пунктуации
# punctuation_mismatch_df_filtered

## Дедубликация примеров

Проверим наличие дубликатов в примерах по признаку `src`:

In [ ]:
duplicates = df_train[df_train.duplicated(subset=['src'], keep=False)]
duplicates

Удалим все дубликаты оставляя первый встреченный пример:

In [ ]:
df_train = df_train.drop_duplicates(subset=['src'], keep='first')
df_train

Проверим дубликаты по признаку `dst`:

In [ ]:
duplicates = df_train[df_train.duplicated(subset=['dst'], keep=False)]
duplicates

## Сохрание очищеной тренировочной части датасета

In [ ]:
# # Обновление датасета после фильтрации
# dataset['train'] = Dataset.from_pandas(df_train, preserve_index=False)

# # Проверка количества строк после фильтрации
# print(f"Number of rows after filtering: {len(dataset['train'])}")


# dataset['train'].to_json('train1.jsonl', orient='records', lines=True, index=False, force_ascii=False)

In [ ]:
dataset